# Analysis - the classifier

A molecule goes in, a set of odor tokens comes out. This reads the committed evaluations in
`output/odor/stage3/` and the ridge baseline in `output/analysis/baseline.json`; it trains nothing.


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SPLITS = ("validation", "test", "dream")

neural = {split: json.loads((ROOT / f"output/odor/stage3/evaluation_{split}.json").read_text()) for split in SPLITS}
baseline = json.loads((ROOT / "output/analysis/baseline.json").read_text())
rows = {split: {row["token"]: row for row in neural[split]["per_token"]} for split in SPLITS}
print("loaded", {split: len(rows[split]) for split in SPLITS}, "scorable tokens per split")

## 1. Headline

Macro ROC-AUC over the tokens that can be scored, meaning those with at least one molecule that carries them and
one that does not. Chance is 0.5.

In [ ]:
headline = pd.DataFrame(
    [
        {
            "split": split,
            "molecules": neural[split]["molecules"],
            "tokens scored": neural[split]["odor_tokens_scored"],
            "ROC-AUC": round(neural[split]["roc_auc_macro"], 3),
            "average precision": round(neural[split]["average_precision_macro"], 3),
            "precision at 5": round(neural[split]["precision_at_5"], 3),
        }
        for split in SPLITS
    ]
).set_index("split")
headline

## 2. Token by token

The macro average is one number over hundreds of tokens, and it hides both ends. This is the distribution behind it
on the test molecules.

In [ ]:
aucs = np.array([row["auc"] for row in neural["test"]["per_token"]])
support = np.array([row["molecules"] for row in neural["test"]["per_token"]])

fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.hist(aucs, bins=40, color="#4472C4")
ax.axvline(np.median(aucs), color="#1F3864", linestyle="--", linewidth=1, label=f"median {np.median(aucs):.3f}")
ax.axvline(0.5, color="#C00000", linestyle="--", linewidth=1, label="chance")
ax.set_xlabel("ROC-AUC of one odor token")
ax.set_ylabel("tokens")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

print(f"{len(aucs)} tokens scored, median {np.median(aucs):.3f}, {np.mean(aucs < 0.5):.0%} worse than chance")
print(f"on 1-2 molecules: {(support <= 2).sum()} tokens, median {np.median(aucs[support <= 2]):.3f}")
print(f"on 10 or more:    {(support >= 10).sum()} tokens, median {np.median(aucs[support >= 10]):.3f}")

In [ ]:
worst = sorted(neural["test"]["per_token"], key=lambda row: row["auc"])[:10]
pd.DataFrame(worst).assign(auc=lambda d: d["auc"].round(3)).set_index("token")

## 3. Against the ridge baseline

Ridge on 2,048 Morgan bits, fit on the same 6,344 training molecules for the same 890 tokens, scored the same way.

In [ ]:
values = np.array([[baseline["splits"][s]["roc_auc_macro"], neural[s]["roc_auc_macro"]] for s in SPLITS])
x = np.arange(len(SPLITS))
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(x - 0.19, values[:, 0], width=0.38, label="ridge on Morgan bits", color="#8FAADC")
ax.bar(x + 0.19, values[:, 1], width=0.38, label="the decoder", color="#1F3864")
for i, (ridge, neural_value) in enumerate(values):
    ax.text(i - 0.19, ridge + 0.01, f"{ridge:.3f}", ha="center", fontsize=8)
    ax.text(i + 0.19, neural_value + 0.01, f"{neural_value:.3f}", ha="center", fontsize=8)
ax.set_xticks(x, SPLITS)
ax.set_ylim(0.5, 0.95)
ax.set_ylabel("macro ROC-AUC")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

for split, (ridge, neural_value) in zip(SPLITS, values, strict=True):
    print(f"{split:10s} ridge {ridge:.3f}  decoder {neural_value:.3f}  difference {neural_value - ridge:+.3f}")

## Reading.

The macro average is 0.821 on the free splits, and the median token is far above it at 0.877. The gap is a long
tail: one token in ten is ranked worse than chance, and almost all of them sit on one or two molecules, where a
single ordering decides the number.

The frequent tokens are not the easy ones. The 44 that appear on ten or more test molecules have a median of
0.849, below the 0.909 of the singletons. A token with many molecules can be scored properly and is; a token with
one is a coin flip the average cannot correct.

Against the baseline the decoder earns its place: 0.832 against 0.725 on validation, 0.811 against 0.736 on test,
0.776 against 0.670 on DREAM. The margin is between seven and eleven points, and it does not shrink on the
hardest set.

## Verdict

The classifier is clearly better than ridge on fingerprints, and it is not uniformly good: it is good on the
tokens that have enough molecules to learn and to measure, and close to useless on the long tail. `precision at 5`
of 0.38 is the honest headline, because the average of 0.8 is carried by hundreds of rare tokens.
